# L41 - Building a Gym-Compatible SimPy Environment

**Learning objectives**
- Inspect the `SimPyEnv` base class used in `simdes`.
- Understand the `reset()` and `step()` contract expected by Gymnasium.
- Build a tiny SimPy-backed environment that advances in decision intervals.
- Identify the extra engineering needed for a full clinic wrapper.

In [ ]:
import gymnasium as gym
import numpy as np
import simpy

from simdes.envs.base_env import SimPyEnv
from simdes.envs.clinic_env import ClinicEnv

env = ClinicEnv(sim_time=120, max_nurses=4, seed=7)
obs, info = env.reset(seed=7)
print('observation_space:', env.observation_space)
print('action_space:', env.action_space)
print('initial observation:', obs)

## A tiny fully-steppable example

The package-level clinic wrapper is still a teaching scaffold. To make the API concrete, the next cell builds a minimal SimPy environment with a queue, an arrival process, a service process, and interval-based actions.

In [ ]:
class ToySimPyQueueEnv(SimPyEnv):
    def __init__(self, sim_time=20.0, decision_interval=2.0, seed=None):
        super().__init__(sim_time=sim_time, seed=seed)
        self.decision_interval = decision_interval
        self.observation_space = gym.spaces.Box(
            low=np.array([0.0, 0.0], dtype=np.float32),
            high=np.array([20.0, 1.0], dtype=np.float32),
            dtype=np.float32,
        )
        self.action_space = gym.spaces.Discrete(2)

    def _build_sim(self):
        self.queue = 0
        self.capacity = 1
        self._env.process(self._arrival_process())
        self._env.process(self._service_process())

    def _arrival_process(self):
        while True:
            yield self._env.timeout(1.0)
            self.queue = min(20, self.queue + int(self._rng.poisson(1.2)))

    def _service_process(self):
        while True:
            yield self._env.timeout(1.0)
            self.queue = max(0, self.queue - self.capacity)

    def _get_obs(self):
        return np.array([self.queue, min(self._env.now / self.sim_time, 1.0)], dtype=np.float32)

    def _step_sim(self, action):
        self.capacity = 1 + int(action)
        target = min(self.sim_time, self._env.now + self.decision_interval)
        self._env.run(until=target)
        return -(self.queue + 0.25 * int(action))

In [ ]:
toy_env = ToySimPyQueueEnv(seed=11)
obs, info = toy_env.reset(seed=11)
trajectory = []
done = False
while not done:
    action = 1 if obs[0] >= 3 else 0
    obs, reward, terminated, truncated, _ = toy_env.step(action)
    trajectory.append((toy_env._env.now, obs[0], action, reward))
    done = terminated or truncated

trajectory[:10]

## Try It Yourself

1. Add a third observation component for current capacity.
2. Change the reward so that queueing and staffing cost are weighted separately.
3. List the additional state needed to make a clinic wrapper genuinely useful for RL training.